In [1]:
## imports 
import pandas as pd
import numpy as np
import yaml
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# comment these out if you don't have plotnine--not essential here/only used once
import matplotlib.pyplot as plt
# import plotnine
# from plotnine import *

## way to connect to mysql 
## if you need to install
## uncomment this line:
#! pip install mysql-connector-python
import mysql.connector

## function to feed path name to load
## credentials
def load_creds(path: str):
    with open(path, 'r') as stream:
        try:
            creds = yaml.safe_load(stream)
        except yaml.YAMLError as exc:
            print(exc)
    return(creds)

pd.options.display.max_rows = 999
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# Preliminary: define connection and read sample of data

In [7]:
## read in creds; change the path name if stored
## elsewhere
creds = load_creds("09_db_cred.yaml")

In [9]:
## connect to the database
cnx = mysql.connector.connect(user=creds['practice_database']['user'], 
                            password=creds['practice_database']['password'],
                            port=creds['practice_database']['port'],
                            database= creds['practice_database']['database'],
                            host = creds['practice_database']['host'])
cnx

# Activity 1

1. Create a new column -- `in_chicago` when pulling from the `caseinit` table that takes on the value of "YES" if INCIDENT_CITY = Chicago; "NO" otherwise (which represents incidents in Cook County suburbs outside the city limits);  and pull the table. Use `crosstabs` to confirm that this worked
2. Repeat step 1 but also filter out blank strings (`INCIDENT_CITY` == "")
3. Use `where` to row filter to initiations in Chicago and use group by to find the count of cases diverted and not diverted (`is_in_diversion`); pull the table with those counts
4. Modify the query in step 3 to find the proportion of cases in chicago diverted (hint you made need to use case when in a subquery)
5. Modify the query in step 4 to find the proportion of cases in chicago versus cases not in chicago sent to diversion 


In [47]:
chicago_q = """
SELECT INCIDENT_CITY,
       CASE
         WHEN INCIDENT_CITY = 'Chicago'
         THEN 'Yes'
         ELSE 'No'
       END AS in_chicago
FROM   caseinit 
"""

compare_charge_d = pd.read_sql_query(chicago_q, cnx)

In [55]:
compare_charge_d

,INCIDENT_CITY,in_chicago
0,,No
1,,No
2,,No
3,,No
4,,No
...,...,...
272289,Chicago,Yes
272290,Evergreen Park,No
272291,Chicago,Yes
272292,Chicago,Yes


In [57]:
chicago_q_2 = """
SELECT INCIDENT_CITY,
       CASE
         WHEN INCIDENT_CITY = 'Chicago'
         THEN TRUE
         ELSE FALSE
       END AS in_chicago
FROM   caseinit 
WHERE INCIDENT_CITY <> ""
"""

compare_charge_d_2 = pd.read_sql_query(chicago_q_2, cnx)

In [61]:
compare_charge_d_2

,INCIDENT_CITY,in_chicago
0,Oak Park,0
1,Harvey,0
2,Morton Grove,0
3,Chicago,1
4,Chicago,1
...,...,...
251732,Chicago,1
251733,Evergreen Park,0
251734,Chicago,1
251735,Chicago,1


In [63]:
sample_case_q = """ 
SELECT is_in_diversion, COUNT(*) AS diversion_count
FROM caseinit
WHERE incident_city = 'Chicago'
GROUP BY is_in_diversion
"""
# read into sample table
read_sample_d = pd.read_sql_query(sample_case_q, cnx)
read_sample_d

,is_in_diversion,diversion_count
0,False,167171
1,True,6402


In [65]:
divert_filter = """
SELECT is_in_diversion, 
    COUNT(*) AS div_count
FROM   caseinit 
WHERE INCIDENT_CITY = 'Chicago'
GROUP BY is_in_diversion
LIMIT 100
"""

activity13 = pd.read_sql_query(divert_filter, cnx)

activity13

,is_in_diversion,div_count
0,False,167171
1,True,6402


In [67]:
prop_div = """
SELECT 
    SUM(CASE WHEN is_in_diversion = 'True' THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS proportion_diverted
FROM (
    SELECT *,
           CASE 
               WHEN INCIDENT_CITY = 'Chicago' THEN 'YES' 
               ELSE 'NO' 
           END AS in_chicago2
    FROM caseinit
) AS tmp
WHERE in_chicago2 = 'YES';
"""

prop_div_d = pd.read_sql_query(prop_div, cnx)
prop_div_d

,proportion_diverted
0,0.03688


In [69]:
prop_all = """
SELECT 
    SUM(CASE WHEN is_in_diversion = 'True' THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS proportion_diverted
FROM (
    SELECT *,
           CASE 
               WHEN INCIDENT_CITY = 'Chicago' THEN 'YES' 
               ELSE 'NO' 
           END AS in_chicago2
    FROM caseinit
) AS tmp
GROUP_BY in_chicago
"""

prop_all_2 = pd.read_sql_query(prop_all,cnx)
prop_all_2

DatabaseError: Execution failed on sql '
SELECT 
    SUM(CASE WHEN is_in_diversion = 'True' THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS proportion_diverted
FROM (
    SELECT *,
           CASE 
               WHEN INCIDENT_CITY = 'Chicago' THEN 'YES' 
               ELSE 'NO' 
           END AS in_chicago2
    FROM caseinit
) AS tmp
GROUP_BY in_chicago
': 1064 (42000): You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near 'GROUP_BY in_chicago' at line 11

# Activity 2 

1. Use the following crosswalk and the `CASE` variable in the `divert` table to create a new variable `DIVERSION_PROGRAM_TEXT` that spells out the diversion programs
    - DC: Drug Court

    - DDPP: Drug Deferred Prosecution

    - DS: Drug School

    - RJCC: Restorative Justice

    - MHC: Mental Health Court

    - VC: Veteran Court

2. Build on the query from step 1 to filter to Narcotics as the `UPDATED_OFFENSE_CATEGORY` and Black or White defendants (based on race in the diversions table) (hint: you'll need to join with the caseinit table based on case_id and case_participant_id, you can do a inner join to keep only those diverted). Select the case_id, case_participant_id, case, race, and diversion_program_text columns

In [ ]:
# your code here 1

In [ ]:
# your code here 2